# v8 — Scaling Up + Checkpointing

This builds directly on **v7**. Steps 1–11 below carry over as-is (setup, `Head`, `MultiHeadAttention`, `FeedForward`, `TransformerBlock`).

What's new:

1. Bigger hyperparameters: `emb_size: 32 → 128`, `block_size: 8 → 128`, `batch_size: 32 → 64`, `learning_rate: 1e-3 → 3e-4`, plus two new ones that make depth/width configurable instead of hardcoded: `num_layers` (number of transformer blocks) and `num_heads`.
2. `BigramLanguageModel.blocks` is now built from `num_layers` / `num_heads` via a list comprehension, instead of 4 hardcoded `TransformerBlock(emb_size, num_heads=8)` lines.
3. **Checkpointing**: during training, whenever validation loss improves, we save the model/optimizer state to disk so the best checkpoint isn't lost.

## 1. Imports

`os` is new — needed to build the checkpoint file path.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

## 2. Hyperparameters

Bigger model, plus `num_layers` / `num_heads` (now configurable instead of hardcoded) and bookkeeping for checkpointing (`out_dir`, `best_val_loss`).

In [ ]:
out_dir = 'output'   # NEW: where to save checkpoints
emb_size = 128        # embedding size for each token
batch_size = 64        # how many independent sequences will we process in parallel?
block_size = 128       # what is the maximum context length for predictions?
max_iters = 10000    # maximum number of iterations for training
eval_interval = 500  # interval for evaluating the model
learning_rate = 3e-4  # learning rate for training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200     # number of iterations for evaluation
num_layers = 4        # NEW: number of transformer blocks
num_heads = 8          # NEW: number of heads in the multi-head attention
seed = 42            # seed for reproducibility
best_val_loss = float('inf') # NEW: tracks the best validation loss seen so far

torch.manual_seed(seed)

## 3. Load the dataset

In [ ]:
with open('./data/harry_potter.txt', encoding='utf-8') as f:
    text = f.read()

print(f"length of dataset in characters: {len(text)}")
print(text[:500])

## 4. Tokenization: characters as tokens

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("Hello there!"))
print(decode(encode("Hello there!")))

## 5. Train / validation split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(data.shape, data.dtype)
print(data[:100])

## 6. Batching

In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

## 7. Estimating loss

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## 8. Self-attention head (recap)

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        key = self.key(x) # (B,T,H)
        query = self.query(x) # (B,T,H)

        # compute the attention weights
        dot_products = query @ key.transpose(-2,-1) * C**-0.5 # (B,T,H) @ (B,H,T) = (B,T,T) # scale by sqrt(d_k)
        dot_products = dot_products.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # mask out the upper triangular part
        weights = F.softmax(dot_products, dim=-1) # (B,T,T) Apply softmax to get the attention weights

        # apply the attention weights to the values
        value = self.value(x) # (B,T,H)
        out = weights @ value # (B,T,T) @ (B,T,H) = (B,T,H)
        return out

## 9. Multi-head attention (recap)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_size, emb_size)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        x = self.proj(x)
        return x

## 10. Feed-forward layer (recap)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_size, 4 * emb_size),
            nn.ReLU(),
            nn.Linear(4 * emb_size, emb_size),
        )

    def forward(self, x):
        return self.net(x)

## 11. Transformer block (recap)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embd_size, num_heads):
        super().__init__()
        head_size = embd_size // num_heads
        self.att_head = MultiHeadAttention(num_heads, head_size)
        self.feedforward = FeedForward(emb_size)
        self.ln1 = nn.LayerNorm(emb_size)
        self.ln2 = nn.LayerNorm(emb_size)

    def forward(self, x):
        x = x + self.att_head(self.ln1(x))
        x = x + self.feedforward(self.ln2(x))
        return x

## 12. The model — what's new

`self.blocks` is now built from `num_layers` / `num_heads` instead of 4 hardcoded lines, so changing the model's depth or width is a one-line config change.

**In class:** fill in the part marked **NEW** below.

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, emb_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
        # we will get to the position embeddings later
        self.position_embedding_table = nn.Embedding(block_size, emb_size)
        # NEW: build num_layers TransformerBlocks instead of hardcoding 4
        self.blocks = ...
        self.linear_head = nn.Linear(emb_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        logits = self.linear_head(x)  # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the block size
            idx_cond = idx[:, -block_size:] # (B, T)

            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## 13. Sanity check: untrained generation

`model_args` is new — it's just a snapshot of the architecture config, bundled up so it can be saved alongside the weights in a checkpoint.

In [ ]:
model = BigramLanguageModel(vocab_size, emb_size)
model = model.to(device)

# NEW: a record of the architecture, so we know how to rebuild the model from a checkpoint later
model_args = dict(n_layer=num_layers, n_head=num_heads, n_embd=emb_size, block_size=block_size,
                   bias=False, vocab_size=None)

# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

## 14. Training the model — what's new

Same training loop as before, plus checkpointing: whenever validation loss improves at an eval step, save the model and optimizer state to disk.

**In class:** fill in the parts marked **NEW** below.

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # NEW: whenever validation loss improves, checkpoint the model + optimizer state
        if ...:
            best_val_loss = ...
            checkpoint = {
                'model': ...,
                'optimizer': ...,
                'model_args': model_args,
                'iter_num': iter,
                'best_val_loss': best_val_loss,
            }
            print(f"saving checkpoint to {out_dir}")
            torch.save(checkpoint, os.path.join(out_dir, 'v8_ckpt.pt'))

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## 15. Generate text from the trained model

In [ ]:
print('''\n##########################################
# Let's generate some Harry Potter text! #
##########################################''')
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))